In [2]:
!kaggle datasets download kazanova/sentiment140

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
sentiment140.zip: Skipping, found more recently modified local copy (use --force to force download)


In [3]:
!unzip sentiment140.zip

Archive:  sentiment140.zip
replace training.1600000.processed.noemoticon.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: training.1600000.processed.noemoticon.csv  


**1. Dataset overview**

In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
from matplotlib import pyplot as plt
import re
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.width', 500)
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler, RobustScaler

In [5]:
df = pd.read_csv('/content/training.1600000.processed.noemoticon.csv', encoding='latin1', header=None)
df.head()

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [6]:
df = df.rename(columns={
    0: 'target',
    1: 'ids',
    2: 'date',
    3: 'flag',
    4: 'user',
    5: 'text'
})

print(df.head())

   target         ids                          date      flag             user                                               text
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY    scotthamilton  is upset that he can't update his Facebook by ...
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY         mattycus  @Kenichan I dived many times for the ball. Man...
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY          ElleCTF    my whole body feels itchy and like its on fire 
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY           Karoli  @nationwideclass no, it's not behaving at all....


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 6 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   target  1600000 non-null  int64 
 1   ids     1600000 non-null  int64 
 2   date    1600000 non-null  object
 3   flag    1600000 non-null  object
 4   user    1600000 non-null  object
 5   text    1600000 non-null  object
dtypes: int64(2), object(4)
memory usage: 73.2+ MB


In [8]:
df = df[["text"]].copy()
print(f"Umumi tvit sayi: {len(df):,}")
print("\nTemizlenmemish ilk 5 tvit:")
print(df.head(5))

Umumi tvit sayi: 1,600,000

Temizlenmemish ilk 5 tvit:
                                                text
0  @switchfoot http://twitpic.com/2y1zl - Awww, t...
1  is upset that he can't update his Facebook by ...
2  @Kenichan I dived many times for the ball. Man...
3    my whole body feels itchy and like its on fire 
4  @nationwideclass no, it's not behaving at all....


**2. Text Cleaning**

In [9]:
def clean_tweet(text: str) -> str:
    # 1. Butun metni kichik herflere chevirmek (Lowercasing)
    text = text.lower()

    # 2. Linkleri temizlemek uchun (http, https, www)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # 3. User tag-lerini temizlemek (@username)
    text = re.sub(r"@\w+", "", text)

    # 4. HTML xususi simvollarini temizlemek (&amp;, &lt;, &gt; və s.)
    text = re.sub(r"&[a-zA-Z]+;", "", text)

    # 5. Herfler ve boshluqlarini bashqa butun simvollari silirik
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # 6. Artiq bosluqları ve setir bashi/sonu boshluqlari silirik
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [10]:
df = df.sample(n=600000, random_state=42).reset_index(drop=True)

In [11]:
df["clean_text"] = df["text"].apply(clean_tweet)

In [12]:
df_sample = df[df["clean_text"].str.len() > 3].reset_index(
    drop=True
)

In [13]:
print("\n--- evvel ve sonra muqayise ---")
for idx, row in df_sample.head(5).iterrows():
    print(f"Orijinal : {row['text']}")
    print(f"Temizlenmish: {row['clean_text']}")
    print("-" * 50)


--- evvel ve sonra muqayise ---
Orijinal : @chrishasboobs AHHH I HOPE YOUR OK!!! 
Temizlenmish: ahhh i hope your ok
--------------------------------------------------
Orijinal : @misstoriblack cool , i have no tweet apps  for my razr 2
Temizlenmish: cool i have no tweet apps for my razr
--------------------------------------------------
Orijinal : @TiannaChaos i know  just family drama. its lame.hey next time u hang out with kim n u guys like have a sleepover or whatever, ill call u
Temizlenmish: i know just family drama its lamehey next time u hang out with kim n u guys like have a sleepover or whatever ill call u
--------------------------------------------------
Orijinal : School email won't open  and I have geography stuff on there to revise! *Stupid School* :'(
Temizlenmish: school email wont open and i have geography stuff on there to revise stupid school
--------------------------------------------------
Orijinal : upper airways problem 
Temizlenmish: upper airways problem
----

**3. Sream Simulyasiya** *tam olaraq bu istənildi yoxsa yox başa düşməmişəm. Süni intellekt köməyi ilə yazmışam.*

In [14]:
import time

def simulate_tweet_stream(df, delay_seconds=1.0):
    """
    Pandas DataFrame-dəki tvitləri canlı axın kimi saniyə-saniyə ötürən generator.
    """
    for index, row in df.iterrows():
        # Təmizlənmiş tviti ötürür (yield) və növbəti tvitə keçməzdən əvvəl gözləyir
        yield row['clean_text']
        time.sleep(delay_seconds)


In [15]:
print("Simulyasiya edilmiş tweet axını (ilk 5 tweet):")
for i, tweet in enumerate(simulate_tweet_stream(df_sample.head(5), delay_seconds=0.5)):
    print(f"Tweet {i+1}: {tweet}")

Simulyasiya edilmiş tweet axını (ilk 5 tweet):
Tweet 1: ahhh i hope your ok
Tweet 2: cool i have no tweet apps for my razr
Tweet 3: i know just family drama its lamehey next time u hang out with kim n u guys like have a sleepover or whatever ill call u
Tweet 4: school email wont open and i have geography stuff on there to revise stupid school
Tweet 5: upper airways problem


**4. NLP Modelinin Tətbiqi (Inference)**

In [17]:
from transformers import pipeline
#bu kod SI-den yazilib

print("NLP Modeli yuklenir...")

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
candidate_labels = ["Sports", "Politics", "Tech", "Entertainment"]


def predict_tweet_topic(text):
    """Tviti təhlil edib mövzunu və əminlik faizini qaytarır."""
    res = classifier(text, candidate_labels)

    topic = res['labels'][0]
    confidence = res['scores'][0] * 100

    return topic, confidence


# 2. Canli tvit axininin analizi (İlk 7 tvit)
print("\n--- Canli Axin ve Tesnifat Bashladi ---\n")

tweet_stream = simulate_tweet_stream(df_sample, delay_seconds=0.5)

# 'enumerate' istifade ederek ilk 7 tviti sirayla temiz shekilde emal edirmish. Bu hisseni de SI-den sorushdum.
for i, live_tweet in enumerate(tweet_stream, start=1):
    topic, conf = predict_tweet_topic(live_tweet)

    print(f"[{i}] Tvit: {live_tweet}")
    print(f"    Movzu: {topic} (Eminlik: {conf:.2f}%)\n")

    if i == 7:
        break


NLP Modeli yuklenir...


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


--- Canli Axin ve Tesnifat Bashladi ---

[1] Tvit: ahhh i hope your ok
    Movzu: Tech (Eminlik: 50.20%)

[2] Tvit: cool i have no tweet apps for my razr
    Movzu: Tech (Eminlik: 63.96%)

[3] Tvit: i know just family drama its lamehey next time u hang out with kim n u guys like have a sleepover or whatever ill call u
    Movzu: Entertainment (Eminlik: 41.92%)

[4] Tvit: school email wont open and i have geography stuff on there to revise stupid school
    Movzu: Tech (Eminlik: 46.11%)

[5] Tvit: upper airways problem
    Movzu: Tech (Eminlik: 80.62%)

[6] Tvit: going to miss pastors sermon on faith
    Movzu: Tech (Eminlik: 52.82%)

[7] Tvit: on lunchdj should come eat with me
    Movzu: Entertainment (Eminlik: 48.32%)



In [16]:
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install --upgrade transformers


Looking in indexes: https://download.pytorch.org/whl/cu118
